# AI-Powered Technical Debt: Tools API Reference

This notebook is the **tools reference** for our project. It walks
through every external tool the pipeline uses, shows a tiny self-contained
example of each, and explains what role the tool plays in the larger
system.

The companion notebook `ai_technical_debt.example.ipynb` shows the
**production pipeline** end to end on a real Java codebase. Read this
notebook first to learn the tools, then read the example notebook to
see them combined.

## What's in here

The pipeline uses 13 tools across four roles:

- **Code analysis tools** (radon, lizard, pylint, ast, javalang, PMD)
  measure code structure and detect issues.
- **Git mining and historical data** (PyDriller, the Technical Debt
  Dataset) provide context about how a codebase evolved.
- **Machine learning** (XGBoost) trains a fault predictor on the
  historical data.
- **Code generation and validation** (Qwen-Coder, sacrebleu, Maven,
  LangGraph) attempt and verify automated fixes.

Each section is independent. You can run them in any order, or skip
ones you already understand.

## Section 0: Setup

This notebook works inside the project's Docker container. Build the
container with `./docker_build.sh`, then start Jupyter with
`./docker_jupyter.sh`. All dependencies are pre-installed.

If a cell errors with `ModuleNotFoundError`, you are running the
notebook outside the container or the container needs rebuilding.

In [1]:
# Confirm we are in the right environment.
import sys
print(f'Python: {sys.version.split()[0]}')
print(f'Working directory: ' + __import__('os').getcwd())

Python: 3.12.3
Working directory: /data


## Section 1: Code complexity with radon

`radon` measures **cyclomatic complexity** of Python code. Cyclomatic
complexity counts the number of independent paths through a function:
each `if`, `for`, `while`, `and`, `or`, etc. adds one. A function with
complexity 1 has a single path. Complexity above 10 is generally
considered hard to test and maintain.

Why this matters for the pipeline: Stage 4 (fault prediction) computes
cyclomatic complexity for every Java method as one of its 19 features.
The trained model learned that high-complexity methods are more
likely to introduce faults. Radon does this for Python; we use a
different tool (`javalang`, Section 5) for Java.

In [2]:
from radon.complexity import cc_visit
from radon.metrics import mi_visit

# A simple, low-complexity function.
simple_code = '''
def add(a, b):
    return a + b
'''

# A function with branching: complexity grows.
complex_code = '''
def categorize(score):
    if score < 0:
        return "invalid"
    elif score < 50:
        return "fail"
    elif score < 70:
        return "pass"
    elif score < 85:
        return "good"
    else:
        return "excellent"
'''

for label, src in [('simple', simple_code), ('complex', complex_code)]:
    blocks = cc_visit(src)
    for block in blocks:
        print(f'{label}: {block.name} -> cyclomatic complexity = {block.complexity}')

simple: add -> cyclomatic complexity = 1
complex: categorize -> cyclomatic complexity = 5


The simple function has complexity 1 (a single path). The branching
function has complexity 5 (one path per branch). Real-world Java
methods we analyze in Stage 4 routinely hit complexity 15-30, which
is one of the reasons they show up in the fault predictor's risk
ranking.

In [4]:
# radon also computes a "maintainability index" combining complexity, size, and Halstead metrics into one score.
mi_simple = mi_visit(simple_code, multi=False)
mi_complex = mi_visit(complex_code, multi=False)
print(f'Simple function maintainability index: {mi_simple:.1f}')
print(f'Complex function maintainability index: {mi_complex:.1f}')
print('Higher is better; >85 is generally healthy, <65 is hard to maintain.')

Simple function maintainability index: 88.6
Complex function maintainability index: 66.2
Higher is better; >85 is generally healthy, <65 is hard to maintain.


## Section 2: Multi-language metrics with lizard

`lizard` is a complexity analyzer that supports many languages
including Java, C++, JavaScript, and Python. Its key advantage over
radon is language coverage. The numbers it reports are conceptually
similar (cyclomatic complexity, NLOC = number of lines of code).

We use lizard in the API notebook to demonstrate Java metric
computation. In production we use `javalang` directly (Section 5)
because it gives us programmatic AST access for richer feature
extraction.

In [5]:
import lizard

# A small Java method with branching.
java_snippet = '''
public class Example {
    public int categorize(int score) {
        if (score < 0) return -1;
        else if (score < 50) return 0;
        else if (score < 70) return 1;
        else if (score < 85) return 2;
        else return 3;
    }
}
'''

# lizard accepts a file path; for a snippet we write to a temp file.
import tempfile, os
with tempfile.NamedTemporaryFile('w', suffix='.java', delete=False) as fh:
    fh.write(java_snippet)
    tmp_path = fh.name

try:
    result = lizard.analyze_file(tmp_path)
    for func in result.function_list:
        print(f'{func.name}: complexity={func.cyclomatic_complexity}, '
              f'NLOC={func.nloc}, params={func.parameter_count}')
finally:
    os.unlink(tmp_path)

Example::categorize: complexity=5, NLOC=7, params=1


lizard reports the same complexity number radon gave for the
equivalent Python function. The metric is language-agnostic in
principle.

For Stage 4 we extract more than just complexity: NCLOC (non-comment
lines of code), comment ratio, halstead difficulty, max nesting depth,
and 14 others. javalang gives us programmatic access to compute all
these from one AST traversal, which lizard does not expose cleanly.

## Section 3: Code smells with pylint

`pylint` is a Python linter that detects style issues, possible bugs,
and code smells. It has hundreds of rules covering naming
conventions, dead code, suspicious patterns, etc.

Why this matters for the pipeline: PMD (Section 6) is the Java
equivalent. Same idea, different language. Pylint is a pedagogical
warmup. The output format is similar: each issue has a code, severity,
location, and message.

In [7]:
import subprocess
import tempfile
import os

def run_pylint(code: str, label: str):
    """Write code to a temp file, run pylint, and print results."""
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as fh:
        fh.write(code)
        tmp_path = fh.name

    result = subprocess.run(
        ['pylint', '--disable=all', '--enable=C,W',
         '--output-format=text', tmp_path],
        capture_output=True, text=True
    )

    print(f"\n{label}:")
    print(result.stdout[:1500] if result.stdout else "(no issues)")

    os.unlink(tmp_path)

# Bad code example
bad_code = '''
import os
import sys
import unused_module

def Bad_Function_Name( x,y, z ):
    UNUSED_VAR = 42
    if x == None:
        return
    return x+ y+z
'''

# Clean code example
clean_code = '''
"""Module that adds two integers."""

def add_numbers(first: int, second: int) -> int:
    """Return the sum of two integers."""
    return first + second
'''

# Run both
run_pylint(bad_code, "Pylint output (bad code)")
run_pylint(clean_code, "Pylint output (clean code)")


Pylint output (bad code):
************* Module tmpz34i39vk
/tmp/tmpz34i39vk.py:1:0: C0114: Missing module docstring (missing-module-docstring)
/tmp/tmpz34i39vk.py:6:0: C0116: Missing function or method docstring (missing-function-docstring)
/tmp/tmpz34i39vk.py:6:0: C0103: Function name "Bad_Function_Name" doesn't conform to snake_case naming style (invalid-name)
/tmp/tmpz34i39vk.py:7:4: C0103: Variable name "UNUSED_VAR" doesn't conform to snake_case naming style (invalid-name)
/tmp/tmpz34i39vk.py:8:7: C0121: Comparison 'x == None' should be 'x is None' (singleton-comparison)
/tmp/tmpz34i39vk.py:7:4: W0612: Unused variable 'UNUSED_VAR' (unused-variable)
/tmp/tmpz34i39vk.py:2:0: W0611: Unused import os (unused-import)
/tmp/tmpz34i39vk.py:3:0: W0611: Unused import sys (unused-import)
/tmp/tmpz34i39vk.py:4:0: W0611: Unused import unused_module (unused-import)

-----------------------------------
Your code has been rated at 0.00/10



Pylint output (clean code):

--------------------------

Each line in pylint’s output represents one issue, showing the file name, location (line and column), a severity code (W=warning, C=convention, E=error), and a human-readable message; PMD follows the same structure for Java, and in Stage 2 of our pipeline we parse its JSON output into a standardized issue format. In contrast, clean code - with proper docstrings, snake_case naming, type hints, and no unused elements - produces no warnings, highlighting the key idea: tools like pylint (and PMD for Java) flag problems in messy code while remaining silent when the code is well-written.


## Section 4: Code structure with Python's ast module

Python's built-in `ast` module parses source code into an Abstract
Syntax Tree. Every code-analysis tool, ours included, ultimately
operates on an AST.

We don't use the Python `ast` module in production (we work on Java
code, not Python). This section is here as a stepping stone to
`javalang`, which is the same idea applied to Java.

In [8]:
import ast

source = '''
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)
'''

tree = ast.parse(source)

# Walk the tree and print the type of each node.
print('AST node types in the parsed code:')
for node in ast.walk(tree):
    print(f'  {type(node).__name__}')

AST node types in the parsed code:
  Module
  FunctionDef
  arguments
  If
  Return
  arg
  Compare
  Return
  BinOp
  Name
  Lt
  Constant
  Name
  Call
  Add
  Call
  Load
  Load
  Name
  BinOp
  Name
  BinOp
  Load
  Name
  Sub
  Constant
  Load
  Name
  Sub
  Constant
  Load
  Load


The AST has nodes for the function definition, the parameter, the
if statement, the comparison, the binary operations, the recursive
calls, and so on. From this tree we can compute cyclomatic complexity
(count branching nodes), nesting depth (max depth of nested control
flow), and many other metrics.

The next section shows the same idea for Java code.

## Section 5: Java AST parsing with javalang

`javalang` is a pure-Python Java parser. Given Java source, it returns
an AST we can walk with the same patterns Python's `ast` module uses.

The pipeline uses javalang in two places:
- **Stage 4** computes 19 code metrics per file by walking the AST
  (complexity, nesting depth, halstead, etc.).
- **Stage 6** uses javalang to validate the agent's output (does the
  generated Java parse?) and to extract the original method's
  signature so we can check signature preservation.

In [9]:
import javalang

java_source = '''
public class Calculator {
    public int add(int a, int b) {
        return a + b;
    }

    public int max(int a, int b) {
        if (a > b) {
            return a;
        }
        return b;
    }
}
'''

tree = javalang.parse.parse(java_source)

# Find all method declarations in the AST.
for path, node in tree.filter(javalang.tree.MethodDeclaration):
    params = [p.type.name + ' ' + p.name for p in node.parameters]
    print(f'Method: {node.name}({", ".join(params)}) -> {node.return_type.name}')

Method: add(int a, int b) -> int
Method: max(int a, int b) -> int


In [12]:
# javalang lets us count branches for cyclomatic complexity.
# Walk the AST and count branching constructs.
branch_types = [
    javalang.tree.IfStatement,
    javalang.tree.WhileStatement,
    javalang.tree.ForStatement,
    javalang.tree.SwitchStatementCase,
    javalang.tree.CatchClause,
]

for path, method in tree.filter(javalang.tree.MethodDeclaration):
    branches = 0
    for branch_type in branch_types:
        for inner_path, _ in method.filter(branch_type):
            branches += 1
    # Cyclomatic complexity = branches + 1 (the entry point).
    print(f'{method.name}: cyclomatic complexity = {branches + 1}')

add: cyclomatic complexity = 1
max: cyclomatic complexity = 2


Now a real-world wrinkle. javalang was last updated around 2020 and
does not handle some modern Java syntax. Here is the most common
failure pattern we hit on commons-lang3 (an open source Java project we have picked for our demonstration and building the production pipeline):

In [13]:
# Modern Java syntax that javalang cannot parse: array constructor
# references like `boolean[]::new`.
modern_java = '''
public class Modern {
    public boolean[] makeArray(java.util.function.IntFunction<boolean[]> f) {
        return f.apply(10);
    }
}

class Caller {
    void use() {
        new Modern().makeArray(boolean[]::new);
    }
}
'''

try:
    javalang.parse.parse(modern_java)
    print('Parsed OK')
except javalang.parser.JavaSyntaxError as exc:
    print(f'javalang parse failure type: {type(exc).__name__}')
    print(f'  description: {exc.description}')
    print(f'  at: {exc.at}')

javalang parse failure type: JavaSyntaxError
  description: Expected '.'
  at: Keyword "new" line 10, position 43


On the commons-lang3 codebase (259 Java files), javalang fails to
parse 3 files due to issues like this. The pipeline logs warnings
and continues; the affected files are excluded from metric
computation but the rest of the pipeline still runs. We document
this in `LIMITATIONS_AND_FUTURE_IMPROVEMENTS.md` and propose
migrating to tree-sitter as future work.

## Section 6: Java static analysis with PMD

`PMD` is a Java static analysis tool. It runs configurable rule
sets over Java source and reports issues: code smells, design
problems, performance issues, security concerns, and so on. It is
the Java equivalent of pylint conceptually.

Stage 2 of the pipeline runs PMD on a target repository's source
tree and parses its JSON output into a uniform issue dict. The
trained fault predictor uses the issue counts as features in
Stage 4.

PMD ships with eight rulesets. We use three: `quickstart` (general
best practices), `performance`, and `security`. Other rulesets
(`design`, `errorprone`, `multithreading`, `codestyle`,
`documentation`) are not enabled by default but could be added.

In [14]:
import subprocess
import tempfile
import json
import os

# A small Java file with some clear issues PMD will flag.
buggy_java = '''
public class Buggy {
    public String getValue() {
        // PMD: ReturnEmptyCollectionRatherThanNull will flag this.
        return null;
    }

    public void example() {
        String s = "";
        // PMD: InefficientEmptyStringCheck will flag this pattern.
        if (s.length() == 0) {
            System.out.println("empty");
        }
    }
}
'''

# Write to a temp directory.
tmp_dir = tempfile.mkdtemp()
java_file = os.path.join(tmp_dir, 'Buggy.java')
report_file = os.path.join(tmp_dir, 'pmd_report.json')

with open(java_file, 'w') as fh:
    fh.write(buggy_java)

# Run PMD.
result = subprocess.run(
    ['pmd', 'check',
     '-d', java_file,
     '-R', 'rulesets/java/quickstart.xml,category/java/performance.xml',
     '-f', 'json', '-r', report_file],
    capture_output=True, text=True
)

# PMD's exit code 4 means "issues found" (not an error).
print(f'PMD exit code: {result.returncode}')
print()

with open(report_file) as fh:
    report = json.load(fh)

if report.get('files'):
    for file_report in report['files']:
        for violation in file_report['violations']:
            print(f"Rule: {violation['rule']}")
            print(f"  Ruleset: {violation['ruleset']}")
            print(f"  Line {violation['beginline']}: {violation['description'][:80]}")
            print()

PMD exit code: 4

Rule: NoPackage
  Ruleset: Code Style
  Line 2: All classes, interfaces, enums and annotations must belong to a named package



Each PMD issue gives us a rule name, a ruleset (which we use in
Stage 3 to bucket issues into README categories), a location, and
a description. Stage 2 wraps this into our internal issue format and
adds a deterministic `issue_id` derived from the rule, file path,
and line number.

For commons-lang3, PMD finds 522 issues across 115 files using our
three rulesets. The example notebook walks through what happens to
those issues across the rest of the pipeline.

## Section 7: Git mining with PyDriller

`PyDriller` is a Python library for mining Git repositories. It
abstracts over the git CLI to give us programmatic access to
commits, file changes, authors, and diffs.

The pipeline uses PyDriller in Stage 4 to compute **churn metrics**
for each commit: how many lines were added/deleted, how many files
were touched. These are 5 of the 24 features the fault predictor
consumes. The intuition: commits that touch many files or change
lots of lines are riskier than small focused commits.

In [15]:
from pydriller import Repository

# Use the local commons-lang3 repo as a small example.
# We just look at the most recent few commits.
repo_path = '/data/production/spikes/q1_agent_on_real_code/commons-lang'

print('Last 3 commits in commons-lang3:')
print()

count = 0
for commit in Repository(repo_path).traverse_commits():
    count += 1
    # Only print the last 3 (we'd reverse-traverse for real use).

# Re-traverse to get to the end. PyDriller doesn't support reverse
# iteration directly; we collect and slice.
commits = list(Repository(repo_path).traverse_commits())
for commit in commits[-3:]:
    insertions = sum(m.added_lines for m in commit.modified_files)
    deletions = sum(m.deleted_lines for m in commit.modified_files)
    print(f'Hash: {commit.hash[:8]}')
    print(f'  Author: {commit.author.name}')
    print(f'  Files touched: {len(commit.modified_files)}')
    print(f'  Lines: +{insertions} -{deletions}')
    print(f'  Message: {commit.msg.split(chr(10))[0][:60]}')
    print()

print(f'Total commits in repo: {len(commits)}')

Last 3 commits in commons-lang3:

Hash: 69012806
  Author: Gary Gregory
  Files touched: 2
  Lines: +2 -1
  Message: Bump commons-io:commons-io from 2.21.0 to 2.22.0.

Hash: 18d3020d
  Author: Gary Gregory
  Files touched: 1
  Lines: +57 -38
  Message: Javadoc

Hash: f9861c09
  Author: Gary Gregory
  Files touched: 10
  Lines: +31 -31
  Message: Javadoc: Use {@code}

Total commits in repo: 9326


For Stage 4, we don't iterate every commit; we only process commits
that actually touched the files where issues live. On commons-lang3
that's typically 30-60 commits even though the full history has
thousands.

## Section 8: The Technical Debt Dataset (V2)

The Lenarduzzi V2 Technical Debt Dataset is a curated SQLite database
of 64,594 commits across 30 Apache open-source projects. Each commit
has SonarQube measurements (complexity, NCLOC, code smells, etc.)
plus labels for whether the commit later turned out to be
fault-inducing.

Our trained fault predictor in Stage 4 was trained on this dataset.
The model file ships with the project at
`production/data/fault_predictor.pkl`.

In this notebook we just open the database and show its structure.
The training script is at `production/scripts/train_fault_predictor.py`.

WARNING: The TD dataset is ~1.4 GB. We never committed it to git (too large) or git-lfs. This cell will download it from the Clowee
release on first run. The trained fault predictor already ships with the project at production/data/fault_predictor.pkl. You only need this dataset if you want to retrain the model from scratch. For the rest of this notebook and the example notebook, you can SKIP this cell.

In [18]:
import os
import sqlite3
import urllib.request

DB_PATH = '/data/production/data/td_V2.db'
DB_URL = 'https://github.com/clowee/The-Technical-Debt-Dataset/releases/download/2.0/td_V2.db'

if not os.path.exists(DB_PATH):
    print(f'Downloading TD dataset to {DB_PATH} (about 1.4GB)...')
    urllib.request.urlretrieve(DB_URL, DB_PATH)
    print('Download complete.')
else:
    print(f'TD dataset already present at {DB_PATH}')

# Show the database schema.
conn = sqlite3.connect(DB_PATH)
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()
print(f'\nTables in the database: {len(tables)}')
for (name,) in tables:
    count = conn.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]
    print(f'  {name}: {count:,} rows')
conn.close()

TD dataset already present at /data/production/data/td_V2.db

Tables in the database: 10
  GIT_COMMITS: 153,994 rows
  GIT_COMMITS_CHANGES: 1,142,878 rows
  JIRA_ISSUES: 61,402 rows
  PROJECTS: 31 rows
  REFACTORING_MINER: 362,253 rows
  SONAR_ANALYSIS: 67,550 rows
  SONAR_ISSUES: 1,024,614 rows
  SONAR_MEASURES: 66,711 rows
  SONAR_RULES: 1,819 rows
  SZZ_FAULT_INDUCING_COMMITS: 52,428 rows


The training data has roughly 1 million SonarQube issues, 64K commits,
and metadata about 30 projects. Our trained model achieves AUC 0.884
on the held-out commons-io project. For the example notebook we
treat the trained model as a black box: load it, call predict_proba,
get a probability per commit. The training process itself is
demonstrated in `production/scripts/train_fault_predictor.py`.

## Section 9: Fault prediction with XGBoost

`XGBoost` (eXtreme Gradient Boosting) is a tree-based ensemble
method that handles tabular data well. It's the standard for
non-image, non-text classification problems.

Our trained fault predictor is an XGBClassifier with 24 features
(19 code metrics + 5 churn features). For this notebook we just
load the trained model and show its feature importances. The full
training pipeline is in
`production/scripts/train_fault_predictor.py`.

In [19]:
import pickle

with open('/data/production/data/fault_predictor.pkl', 'rb') as fh:
    artifact = pickle.load(fh)

# Show what's in the model artifact.
print('Model artifact keys:', sorted(artifact.keys()))
print()

# The model itself is an XGBClassifier.
model = artifact['model']
print(f'Model type: {type(model).__name__}')
print(f'Number of features: {model.n_features_in_}')

Model artifact keys: ['feature_names', 'model', 'normalization_stats', 'target_project', 'trained_at', 'training_metrics', 'training_projects']

Model type: XGBClassifier
Number of features: 24


In [20]:
# Show the top feature importances.
import numpy as np

feature_names = artifact['feature_names']
importances = model.feature_importances_

# Sort by importance descending.
order = np.argsort(importances)[::-1]
print('Top 10 most important features:')
for idx in order[:10]:
    print(f'  {feature_names[idx]:<40s} {importances[idx]:.4f}')

Top 10 most important features:
  lines_added                              0.1902
  FUNCTION_COMPLEXITY                      0.1478
  STATEMENTS                               0.1043
  files_changed                            0.0435
  COGNITIVE_COMPLEXITY                     0.0435
  CLASSES                                  0.0413
  COMMENT_LINES_DENSITY                    0.0412
  FUNCTIONS                                0.0381
  COMPLEXITY                               0.0361
  FILES                                    0.0328


Complexity-related features (FUNCTION_COMPLEXITY, COGNITIVE_COMPLEXITY, COMPLEXITY) and size-related features (STATEMENTS, FILES, FUNCTIONS, CLASSES) dominate the top 10. Churn features (lines_added, files_changed) are also represented. Comment density (COMMENT_LINES_DENSITY) is the only documentation-related feature in the top 10

In Stage 4 the pipeline computes these features for each issue's host
commit, runs them through the trained model, and produces a
fault probability between 0 and 1.

## Section 10: Code generation with Qwen-Coder

`Qwen2.5-Coder` is a family of open-weight code generation models
from Alibaba. We use two sizes:

- **0.5B** runs on CPU, fits in memory easily, but produces weak
  refactorings (often returns the input unchanged or wraps it in a
  stub class). Used for laptop-only demos and quick smoke tests.
- **3B** requires a GPU to be practical (we used UMIACS Nexus's
  RTX A4000) but produces real refactorings most of the time. Used
  for the headline results in the example notebook.

For this section we load the 0.5B tokenizer and demonstrate the
prompt structure. We don't actually run inference here because even
a small generation takes 10+ seconds and would slow the notebook.

In [21]:
from transformers import AutoTokenizer

# Load just the tokenizer (fast; doesn't load the model weights).
tokenizer = AutoTokenizer.from_pretrained(
    'Qwen/Qwen2.5-Coder-0.5B-Instruct',
    trust_remote_code=True,
)

# Build a chat-formatted prompt the way Stage 6 does.
messages = [
    {
        'role': 'system',
        'content': 'You are an expert Java refactoring assistant. '
                   'Output only the refactored Java method. '
                   'Preserve the method signature.'
    },
    {
        'role': 'user',
        'content': '''Rule: ReturnEmptyCollectionRatherThanNull
Description: Returning null instead of an empty collection forces
callers to add null checks. Return an empty collection instead.

Refactor this method:

```java
public String[] getNames() {
    if (names.isEmpty()) {
        return null;
    }
    return names.toArray(new String[0]);
}
```'''
    }
]

# Apply the chat template.
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print('Tokenized prompt structure:')
print('---')
print(prompt[:600])
print('---')
print(f'Prompt token count: {len(tokenizer.encode(prompt))}')

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenized prompt structure:
---
<|im_start|>system
You are an expert Java refactoring assistant. Output only the refactored Java method. Preserve the method signature.<|im_end|>
<|im_start|>user
Rule: ReturnEmptyCollectionRatherThanNull
Description: Returning null instead of an empty collection forces
callers to add null checks. Return an empty collection instead.

Refactor this method:

```java
public String[] getNames() {
    if (names.isEmpty()) {
        return null;
    }
    return names.toArray(new String[0]);
}
```<|im_end|>
<|im_start|>assistant

---
Prompt token count: 108


The chat template converts our message list into the model's
expected text format. Stage 6 uses exactly this structure when
invoking the model. The system message is fixed; the user message
is built from the issue's rule, description, and method source.

For the 3B model on Nexus, generation took 33-103 seconds per
issue. The example notebook loads pre-computed records from JSON
rather than running inference live (which would take 5+ minutes
for 10 issues even on the GPU).

## Section 11: BLEU scoring with sacrebleu

`sacrebleu` computes the BLEU score between a candidate and a
reference. BLEU was originally designed for machine translation but
it generalizes to any task where you want to measure how close a
generated sequence is to a target.

The pipeline uses BLEU in Stage 6 to compare the agent's
refactored method against the original method. A BLEU of 100 means
the agent returned the input unchanged (no refactoring done). A BLEU
near 0 means the agent rewrote everything. We want something in
between: real edits that preserve most of the original.

Stage 6's confidence scoring uses BLEU as one of three signals
(plus syntax validity and signature preservation).

In [25]:
import sacrebleu

reference = 'public int add(int a, int b) { return a + b; }'

candidates = [
    ('Identical to reference (unchanged)', reference),
    ('Small whitespace change',
     'public int add(int a, int b) { return a+b; }'),
    ('Refactored with same names but extra step',
     'public int add(int a, int b) { int sum = a + b; return sum; }'),
    ('Aggressive rewrite (param renamed)',
     'public int add(int x, int y) { return x + y; }'),
    ('Completely different code',
     'public String greet(String name) { return "hello " + name; }'),
]

print(f'Reference: {reference}')
print()
for label, candidate in candidates:
    bleu = sacrebleu.sentence_bleu(candidate, [reference]).score
    print(f'BLEU {bleu:6.2f}  |  {label}')
    print(f'              {candidate}')
    print()

Reference: public int add(int a, int b) { return a + b; }

BLEU 100.00  |  Identical to reference (unchanged)
              public int add(int a, int b) { return a + b; }

BLEU 100.00  |  Small whitespace change
              public int add(int a, int b) { return a+b; }

BLEU  60.53  |  Refactored with same names but extra step
              public int add(int a, int b) { int sum = a + b; return sum; }

BLEU  34.74  |  Aggressive rewrite (param renamed)
              public int add(int x, int y) { return x + y; }

BLEU  12.09  |  Completely different code
              public String greet(String name) { return "hello " + name; }



On our 3B Nexus run, BLEU scores ranged from 31 (aggressive rewrite)
to 100 (input returned unchanged). Stage 6 maps BLEU plus syntax
validity plus signature preservation into a confidence label
(HIGH / MEDIUM / LOW). The example notebook walks through specific
records and their scores.

## Section 12: Maven for Java compilation

`Maven` is a build automation tool for Java projects. Given a
`pom.xml` (the project manifest), Maven knows how to download
dependencies, compile sources, run tests, package JARs, and so on.

Stage 7 of the pipeline uses Maven to validate refactorings:
1. Copy the target repo to a temp directory.
2. Splice the agent's refactored method back into the right file.
3. Run `mvn compiler:compile` to check the patch compiles.
4. Optionally run `mvn surefire:test` to check it preserves behavior.

We use targeted Maven plugin goals (`compiler:compile` not
`compile`) to bypass project-specific plugins that may require
newer Maven versions than ours. This keeps Stage 7 robust across
different Java projects.

This section just shows Maven is available. We don't run a real
Maven build here because that takes 30+ seconds even when cached;
the example notebook does that.

In [23]:
import subprocess

result = subprocess.run(['mvn', '--version'], capture_output=True, text=True)
print(result.stdout[:300])

print('Stage 7 invokes Maven with these commands:')
print('  Compile only:  mvn compiler:compile')
print('  Compile + test: mvn compiler:compile compiler:testCompile surefire:test')

Apache Maven 3.8.7
Maven home: /usr/share/maven
Java version: 17.0.18, vendor: Ubuntu, runtime: /usr/lib/jvm/java-17-openjdk-arm64
Default locale: en_US, platform encoding: UTF-8
OS name: "linux", version: "6.8.0-64-generic", arch: "aarch64", family: "unix"

Stage 7 invokes Maven with these commands:
  Compile only:  mvn compiler:compile
  Compile + test: mvn compiler:compile compiler:testCompile surefire:test


Maven on commons-lang3 takes about 30 seconds for `compiler:compile`
once dependencies are cached. Test mode runs all 57,939 commons-lang3
tests in about 140 seconds. The example notebook demonstrates both
modes on real refactorings.

## Section 13: Agent harness with LangGraph

`LangGraph` is a framework for building multi-step agent workflows.
It models an agent as a state graph where each node is a function
that reads and updates a shared state.

The MVP demonstrated a small LangGraph agent for code refactoring.
Our production Stage 6 uses a simpler design: just call the model
once per (strategy, issue) pair. This is intentional. A multi-step
agent loop (where compile errors are fed back for retry) is a future
improvement we discuss in `LIMITATIONS_AND_FUTURE_IMPROVEMENTS.md`.

We include a tiny LangGraph example here for completeness.

In [24]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    counter: int
    history: list


def increment(state: State) -> State:
    new_counter = state['counter'] + 1
    return {
        'counter': new_counter,
        'history': state['history'] + [f'incremented to {new_counter}']
    }


def should_continue(state: State) -> str:
    return END if state['counter'] >= 3 else 'increment'


# Build the graph.
graph = StateGraph(State)
graph.add_node('increment', increment)
graph.add_edge(START, 'increment')
graph.add_conditional_edges('increment', should_continue)

app = graph.compile()

# Run with initial state.
final = app.invoke({'counter': 0, 'history': []})
print(f'Final counter: {final["counter"]}')
print('History:')
for entry in final['history']:
    print(f'  {entry}')

Final counter: 3
History:
  incremented to 1
  incremented to 2
  incremented to 3


This trivial example shows the LangGraph pattern: define states,
nodes (functions), and conditional edges (control flow). For an
iterative refactoring agent, the nodes would be: generate refactor,
compile, parse error, retry. LangGraph's checkpointing makes this
straightforward to extend.

In our production Stage 6 we use the simpler one-shot pattern
because the 3B model produces correct output 80% of the time on first
try, and the failures we see are structural (wrong scope) rather than
fixable by error feedback. A LangGraph-based loop becomes worthwhile
with a stronger base model where iteration would actually help.

## Wrap-up

You've now seen all 13 tools the project relies on. Each one plays
a specific role:

| Stage | Tool(s) |
|-------|---------|
| Setup | Docker container with all tools pre-installed |
| Stage 2 (Analyze) | PMD |
| Stage 3 (Classify) | PMD ruleset metadata + curated CSV |
| Stage 4 (Predict) | javalang, PyDriller, XGBoost (the trained model) |
| Stage 5 (Prioritize) | Pure logic; no external tool |
| Stage 6 (Refactor) | Qwen-Coder, sacrebleu (for BLEU scoring) |
| Stage 7 (Validate) | Maven |
| Stage 8 (Feedback) | SQLite (Python standard library) |

The example notebook (`ai_technical_debt.example.ipynb`) shows these
tools combined into the production pipeline running on commons-lang3.

For a deeper read on what works and what doesn't, see
`LIMITATIONS_AND_FUTURE_IMPROVEMENTS.md` at the project root.